# Step 2 ? Analyze Patterns

1. Run the **list cell** to see available models and select one from the dropdown.
2. Run the **configuration cell** to confirm your selection and set other options.
3. Run all remaining cells.

In [ ]:
import os, json
import ipywidgets as widgets
from IPython.display import display

output_dir = "output"
model_names = []
print(f"{'Run name':<45} {'embedding':<20} {'stopwords':<22} {'topics':>6} {'docs':>8}  trained")
print("-" * 120)
if os.path.isdir(output_dir):
    for name in sorted(os.listdir(output_dir)):
        cfg_path = os.path.join(output_dir, name, "run_config.json")
        if os.path.exists(cfg_path):
            with open(cfg_path) as f:
                c = json.load(f)
            print(f"{name:<45} {c.get('embedding','?'):<20} {c.get('stopwords','?'):<22}"
                  f" {c.get('n_topics','?'):>6} {c.get('n_docs','?'):>8}  {c.get('trained_on','?')[:10]}")
            model_names.append(name)
else:
    print("No output/ directory found ? run 01_build_model.ipynb first.")

model_selector = widgets.Dropdown(options=model_names, description="Model:")
display(model_selector)

In [ ]:
# ============================================================
# CONFIGURATION — run all cells after selecting a model above
# ============================================================

# Model selected from the dropdown in the list cell above
MODEL_TO_LOAD = model_selector.value

# Which label columns to cross-tabulate with topics
LABEL_COLUMNS = ["eptype", "mod", "bodp", "yeargr"]

# Which label columns to use in NLP score vs label analysis (can differ from LABEL_COLUMNS)
NLP_LABEL_COLUMNS = ["eptype", "mod", "bodp", "yeargr"]

# NLP scorer modules to apply (each must live in nlp_scripts/ and expose score(text)->float)
NLP_SCORE_SCRIPTS = ["hedge_word_counter", "pi_risk_estimator"]

# Toggle individual analyses
RUN_LABEL_ANALYSIS = True
RUN_SCORE_ANALYSIS = True
RUN_SCORE_BY_LABEL_ANALYSIS = True

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import importlib
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_VIZ = True
except ImportError:
    HAS_VIZ = False
    print("Install matplotlib and seaborn for charts.")

# pandas 3 + pyarrow 24 incompatibility: pyarrow 24 pre-registers extension types
# (e.g. pandas.period) that pandas also tries to register, raising ArrowKeyError.
# Patch register_extension_type once to silently skip already-registered types.
import pyarrow as _pa
if not hasattr(_pa, "_register_patched"):
    _orig_register = _pa.register_extension_type
    def _safe_register(t, _orig=_orig_register, _err=_pa.lib.ArrowKeyError):
        try:
            _orig(t)
        except _err:
            pass
    _pa.register_extension_type = _safe_register
    _pa._register_patched = True
    del _orig_register, _safe_register
del _pa

In [ ]:
run_dir = os.path.join("output", MODEL_TO_LOAD)
cfg_path = os.path.join(run_dir, "run_config.json")

with open(cfg_path) as f:
    run_cfg = json.load(f)

print(f"Loading model: {MODEL_TO_LOAD}")
print(json.dumps(run_cfg, indent=2))

# Re-create the embedding model that was used during training
if run_cfg["embedding"] == "clinical_radiology":
    embedding_model = SentenceTransformer("emilyalsentzer/Bio_ClinicalBERT")
elif run_cfg["embedding"] == "embeddinggemma":
    embedding_model = SentenceTransformer("sentence-transformers/embeddinggemma-300m-medical")
else:
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

topic_model = BERTopic.load(os.path.join(run_dir, "model"), embedding_model=embedding_model)
df = pd.read_parquet(os.path.join(run_dir, "annotated_data.parquet"))
print(f"\nLoaded {len(df)} records. Topics: {sorted(df['topic'].unique())}")
display(df.head())

In [ ]:
# === Label Analysis ===
if RUN_LABEL_ANALYSIS:
    print("=== Topic-Label Relationships ===\n")
    for col in LABEL_COLUMNS:
        if col not in df.columns:
            print(f"  Column '{col}' not found in data — skipping.")
            continue
        dist = df.groupby(["topic", col]).size().unstack(fill_value=0)
        print(f"Distribution of '{col}' across topics:")
        display(dist)
        if HAS_VIZ:
            ax = dist.plot(kind="bar", stacked=True, figsize=(8, 4),
                           title=f"Topic distribution by {col}")
            ax.set_xlabel("Topic")
            ax.set_ylabel("Count")
            plt.tight_layout()
            plt.show()
else:
    print("Label analysis is disabled (RUN_LABEL_ANALYSIS = False).")

In [ ]:
# === Compute NLP Scores ===
active_scores = []
if RUN_SCORE_ANALYSIS or RUN_SCORE_BY_LABEL_ANALYSIS:
    for script_name in NLP_SCORE_SCRIPTS:
        try:
            nlp_mod = importlib.import_module(f"nlp_scripts.{script_name}")
            df[script_name] = df["reptext"].apply(nlp_mod.score)
            active_scores.append(script_name)
            print(f"  Applied: {script_name}")
        except (ImportError, AttributeError) as e:
            print(f"  Warning: could not load '{script_name}': {e}")

# === NLP Score Distributions per Topic ===
if RUN_SCORE_ANALYSIS:
    print("\n=== NLP Score Distributions per Topic ===\n")
    if active_scores:
        print("Mean scores per topic:")
        display(df.groupby("topic")[active_scores].mean().round(3))

        if HAS_VIZ:
            for score_col in active_scores:
                fig, ax = plt.subplots(figsize=(8, 4))
                df.boxplot(column=score_col, by="topic", ax=ax)
                ax.set_title(f"{score_col} distribution per topic")
                ax.set_xlabel("Topic")
                ax.set_ylabel(score_col)
                plt.suptitle("")
                plt.tight_layout()
                plt.show()
else:
    print("Score-by-topic analysis is disabled (RUN_SCORE_ANALYSIS = False).")

In [ ]:
# === NLP Score Distributions per Label ===
if RUN_SCORE_BY_LABEL_ANALYSIS:
    print("=== NLP Score Distributions per Label ===\n")
    if not active_scores:
        print("  No scores available — check NLP_SCORE_SCRIPTS.")
    else:
        for col in NLP_LABEL_COLUMNS:
            if col not in df.columns:
                print(f"  Column '{col}' not found in data — skipping.")
                continue
            print(f"Mean scores by '{col}':")
            display(df.groupby(col)[active_scores].mean().round(3))
            if HAS_VIZ:
                for score_col in active_scores:
                    fig, ax = plt.subplots(figsize=(8, 4))
                    df.boxplot(column=score_col, by=col, ax=ax)
                    ax.set_title(f"{score_col} by {col}")
                    ax.set_xlabel(col)
                    ax.set_ylabel(score_col)
                    plt.suptitle("")
                    plt.tight_layout()
                    plt.show()
else:
    print("Score-by-label analysis is disabled (RUN_SCORE_BY_LABEL_ANALYSIS = False).")